# 如何加快 Input / Output 速度

1. 為什麼大量資料時 `input()` 會慢。
2. `sys.stdin.read()` / `sys.stdin.buffer.read()` 為什麼通常比較快。
3. 大量輸出時，為什麼不要一直 `print()`。
4. 比賽時可以直接背哪幾種模板。


## 0. 先準備測試資料

我們先建立一個文字檔 `input_demo.txt`，裡面放很多數字，每行一個數字。

等一下我們會用不同方法讀這個檔案，並比較速度。

In [1]:
from pathlib import Path

N = 200_000  # 可以改大，例如 1_000_000，但上課示範先不要太大
filename = Path("input_demo.txt")

# 建立測試資料：1, 2, 3, ..., N，每行一個數字
filename.write_text("\n".join(str(i) for i in range(1, N + 1)), encoding="utf-8")

expected_sum = N * (N + 1) // 2
print("資料筆數:", N)
print("正確總和:", expected_sum)
print("檔案大小:", filename.stat().st_size, "bytes")

資料筆數: 200000
正確總和: 20000100000
檔案大小: 1288894 bytes


## 1. 方法一：用 `input()` 一行一行讀

`input()` 的特色：

- 每次只讀一行。
- 每讀一行都有函式呼叫成本。
- 資料量小時沒差，資料量大時會明顯變慢。


```bash
python input_method_demo.py < input_demo.txt
```



In [2]:
import sys
import time
import json
import subprocess
from pathlib import Path

# 產生一個真正會使用 input() 的 Python 程式
input_script = Path("input_method_demo.py")
input_script.write_text(
    'import time\nimport json\n\nstart = time.perf_counter()\ntotal = 0\n\nwhile True:\n    try:\n        line = input().strip()\n    except EOFError:\n        break\n\n    if line:\n        total += int(line)\n\nend = time.perf_counter()\n\nprint(json.dumps({\n    "total": total,\n    "elapsed": end - start\n}))\n',
    encoding="utf-8"
)


def read_with_input(filename):
    # 用真正的 shell 重導向：python input_method_demo.py < input_demo.txt
    command = f'{sys.executable} input_method_demo.py < "{filename}"'
    completed = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
        check=True,
    )
    info = json.loads(completed.stdout)
    return info["total"], info["elapsed"]

print(f"$ {sys.executable} input_method_demo.py < {filename}")
result, elapsed = read_with_input(filename)
print("Method: input()")
print("Sum:", result)
print("Correct:", result == expected_sum)
print(f"Time: {elapsed:.6f} seconds")


$ /opt/anaconda3/bin/python input_method_demo.py < input_demo.txt
Method: input()
Sum: 20000100000
Correct: True
Time: 0.185045 seconds


## 2. 方法二：用 `sys.stdin.read().splitlines()` 一次讀完

這種方法的概念是：

> 不要一行一行慢慢讀，而是一次把所有文字讀進來，再自己切行處理。

常見寫法：

```python
import sys
lines = sys.stdin.read().splitlines()
```

In [3]:
def read_with_stdin_read(filename):
    start = time.perf_counter()
    old_stdin = sys.stdin

    try:
        sys.stdin = open(filename, "r", encoding="utf-8")
        lines = sys.stdin.read().splitlines()
        total = sum(int(line) for line in lines if line.strip())
    finally:
        sys.stdin.close()
        sys.stdin = old_stdin

    end = time.perf_counter()
    return total, end - start

result, elapsed = read_with_stdin_read(filename)
print("Method: sys.stdin.read().splitlines()")
print("Sum:", result)
print("Correct:", result == expected_sum)
print(f"Time: {elapsed:.6f} seconds")

Method: sys.stdin.read().splitlines()
Sum: 20000100000
Correct: True
Time: 0.035231 seconds


## 3. 方法三：用 `sys.stdin.buffer.read()` 讀 bytes

`sys.stdin.buffer.read()` 會先讀成二進位資料，也就是 `bytes`。

例如：

```python
b"123\n456\n789\n"
```

之後再用 `.decode()` 轉成字串。

In [4]:
def read_with_buffer_read(filename):
    start = time.perf_counter()
    old_stdin = sys.stdin

    try:
        sys.stdin = open(filename, "r", encoding="utf-8")
        binary_data = sys.stdin.buffer.read()
        text = binary_data.decode("utf-8")
        lines = text.splitlines()
        total = sum(int(line) for line in lines if line.strip())
    finally:
        sys.stdin.close()
        sys.stdin = old_stdin

    end = time.perf_counter()
    return total, end - start

result, elapsed = read_with_buffer_read(filename)
print("Method: sys.stdin.buffer.read()")
print("Sum:", result)
print("Correct:", result == expected_sum)
print(f"Time: {elapsed:.6f} seconds")

Method: sys.stdin.buffer.read()
Sum: 20000100000
Correct: True
Time: 0.079326 seconds


## 4. 方法四：直接用 `open().read().splitlines()`

這個方法是讀檔案，不是讀標準輸入。

本機測試很方便，例如：

```python
with open("input.txt") as f:
    lines = f.read().splitlines()
```

但是 Online Judge 通常是從標準輸入讀資料，所以正式比賽更常用 `sys.stdin.read()`。

In [5]:
def read_with_open_read(filename):
    start = time.perf_counter()

    with open(filename, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()
        total = sum(int(line) for line in lines if line.strip())

    end = time.perf_counter()
    return total, end - start

result, elapsed = read_with_open_read(filename)
print("Method: open().read().splitlines()")
print("Sum:", result)
print("Correct:", result == expected_sum)
print(f"Time: {elapsed:.6f} seconds")

Method: open().read().splitlines()
Sum: 20000100000
Correct: True
Time: 0.048582 seconds


## 5. 一次比較四種輸入方法

這一格會把四種方法跑在一起，比較容易投影給學生看。

注意：每台電腦結果會不同，不要要求數字完全一樣，重點是看趨勢。

In [6]:
methods = [
    ("input()", read_with_input),
    ("sys.stdin.read().splitlines()", read_with_stdin_read),
    ("sys.stdin.buffer.read()", read_with_buffer_read),
    ("open().read().splitlines()", read_with_open_read),
]

results = []
for name, func in methods:
    total, elapsed = func(filename)
    results.append((name, total, elapsed))

print("Input Benchmark")
print("=" * 60)
for name, total, elapsed in results:
    print(f"Method: {name}")
    print(f"Sum: {total}")
    print(f"Correct: {total == expected_sum}")
    print(f"Time: {elapsed:.6f} seconds")
    print("-" * 60)

Input Benchmark
Method: input()
Sum: 20000100000
Correct: True
Time: 0.181751 seconds
------------------------------------------------------------
Method: sys.stdin.read().splitlines()
Sum: 20000100000
Correct: True
Time: 0.023109 seconds
------------------------------------------------------------
Method: sys.stdin.buffer.read()
Sum: 20000100000
Correct: True
Time: 0.022345 seconds
------------------------------------------------------------
Method: open().read().splitlines()
Sum: 20000100000
Correct: True
Time: 0.021531 seconds
------------------------------------------------------------


## 6. 比賽最常用輸入模板

如果輸入都是數字，最常用的是這個：

```python
import sys

data = list(map(int, sys.stdin.read().split()))
```

`.split()` 會用空白、換行、Tab 自動切開，所以不管輸入長這樣：

```text
5
1 2 3 4 5
```

或是長這樣：

```text
5 1 2 3 4 5
```

都可以讀。

In [7]:
# 模擬 Online Judge 輸入：
# 第一個數字是 n，後面 n 個數字是資料
sample_input = "5\n1 2 3 4 5\n"
Path("sample_input.txt").write_text(sample_input, encoding="utf-8")

old_stdin = sys.stdin
try:
    sys.stdin = open("sample_input.txt", "r", encoding="utf-8")

    data = list(map(int, sys.stdin.read().split()))
    n = data[0]
    arr = data[1:]

    print("n =", n)
    print("arr =", arr)
    print("sum =", sum(arr))
finally:
    sys.stdin.close()
    sys.stdin = old_stdin

n = 5
arr = [1, 2, 3, 4, 5]
sum = 15


## 7. 大量輸出：`print()` vs `sys.stdout.write()`

`print()` 很方便，但大量輸出時會比較慢，因為它會處理：

- 自動換行 `end="\n"`
- 多個參數的分隔 `sep=" "`
- 型別轉字串
- flush 等其他處理

大量輸出時，比較推薦先把答案存起來，最後一次輸出。

In [8]:
import io

M = 200_000

# 為了避免真的在 Notebook 印出 20 萬行，這裡用 io.StringIO 模擬輸出位置

def output_with_print():
    fake_output = io.StringIO()
    old_stdout = sys.stdout
    start = time.perf_counter()

    try:
        sys.stdout = fake_output
        for i in range(M):
            print(i)
    finally:
        sys.stdout = old_stdout

    end = time.perf_counter()
    return end - start


def output_with_write_loop():
    fake_output = io.StringIO()
    start = time.perf_counter()

    for i in range(M):
        fake_output.write(str(i) + "\n")

    end = time.perf_counter()
    return end - start


def output_with_join_once():
    fake_output = io.StringIO()
    start = time.perf_counter()

    ans = [str(i) for i in range(M)]
    fake_output.write("\n".join(ans))

    end = time.perf_counter()
    return end - start

print_time = output_with_print()
write_loop_time = output_with_write_loop()
join_once_time = output_with_join_once()

print("Output Benchmark")
print("=" * 60)
print(f"print() many times:       {print_time:.6f} seconds")
print(f"write() many times:       {write_loop_time:.6f} seconds")
print(f"join + write() one time:  {join_once_time:.6f} seconds")

Output Benchmark
print() many times:       0.067969 seconds
write() many times:       0.017765 seconds
join + write() one time:  0.012223 seconds


## 8. 比賽最常用輸出模板

當你有很多答案要輸出時，不要這樣：

```python
for x in ans:
    print(x)
```

比較建議這樣：

```python
import sys
sys.stdout.write("\n".join(map(str, ans)))
```

In [9]:
ans = [10, 20, 30, 40, 50]

# 正式比賽常用寫法
sys.stdout.write("\n".join(map(str, ans)))

10
20
30
40
50

14

## 9. 課堂練習

### 練習 1

輸入很多整數，請輸出總和。

```text
5
1 2 3 4 5
```

輸出：

```text
15
```

提示：第一個數字是 n，後面才是要加總的資料。

In [10]:
# 練習 1 參考答案
sample_input = "5\n1 2 3 4 5\n"
Path("practice1.txt").write_text(sample_input, encoding="utf-8")

old_stdin = sys.stdin
try:
    sys.stdin = open("practice1.txt", "r", encoding="utf-8")

    data = list(map(int, sys.stdin.read().split()))
    n = data[0]
    nums = data[1:1+n]
    print(sum(nums))
finally:
    sys.stdin.close()
    sys.stdin = old_stdin

15


### 練習 2

輸入很多行，每行一個字串，請把每行前面加上行號後輸出。

輸入：

```text
apple
banana
cat
```

輸出：

```text
1 apple
2 banana
3 cat
```

In [11]:
# 練習 2 參考答案
sample_input = "apple\nbanana\ncat\n"
Path("practice2.txt").write_text(sample_input, encoding="utf-8")

old_stdin = sys.stdin
try:
    sys.stdin = open("practice2.txt", "r", encoding="utf-8")

    lines = sys.stdin.read().splitlines()
    ans = []
    for i, line in enumerate(lines, start=1):
        ans.append(f"{i} {line}")

    sys.stdout.write("\n".join(ans))
finally:
    sys.stdin.close()
    sys.stdin = old_stdin

1 apple
2 banana
3 cat

## 10. 補充：為什麼 `input()` 不一定每次都很慢？

如果只有幾行資料，其實 `input()` 完全可以用，而且比較好懂。

例如：

```python
n = int(input())
a = list(map(int, input().split()))
```

這種在很多題目也很常見。

真正需要換成 `sys.stdin.read()` 的情況通常是：

- 輸入行數很多。
- 題目資料量很大。
- 程式邏輯明明是對的，但一直接近 Time Limit。
- 需要一次處理很多數字或字串。